In [1]:
# Cài đặt thư viện cần thiết (nếu chưa có)
!pip install open_clip_torch matplotlib pillow requests

import torch
import open_clip
from PIL import Image
import requests
from io import BytesIO
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import json

# Kiểm tra GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.4 MB/s eta 0:00:00
Running on: cuda


In [2]:
# Load model một lần để dùng cho cả pipeline
print("Loading BiomedCLIP...")
model, preprocess = open_clip.create_model_from_pretrained('hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')
tokenizer = open_clip.get_tokenizer('hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')
model.to(device)
model.eval()
print("Model loaded successfully!")

Loading BiomedCLIP...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


open_clip_config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

open_clip_pytorch_model.bin:   0%|          | 0.00/784M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Model loaded successfully!


In [3]:
# Cấu hình nhãn (Giống file labels.json)
TRIAGE_LABELS = [
    "Chest X-ray", "Brain MRI", "Abdominal CT", "Histopathology", 
    "Ultrasound", "Dermoscopy", "Gross pathology", "Bone X-ray"
]

def run_triage(image):
    """Phân loại ảnh đầu vào"""
    image_tensor = preprocess(image).unsqueeze(0).to(device)
    texts = tokenizer([f"this is a {l}" for l in TRIAGE_LABELS]).to(device)
    
    with torch.no_grad():
        image_features = model.encode_image(image_tensor)
        text_features = model.encode_text(texts)
        
        # Tính xác suất
        image_features /= image_features.norm(dim=-1, keepdim=True)
        text_features /= text_features.norm(dim=-1, keepdim=True)
        probs = (100.0 * image_features @ text_features.T).softmax(dim=-1)
        
        probs = probs.cpu().numpy()[0]
    
    # Lấy Top-1
    top_idx = probs.argmax()
    return TRIAGE_LABELS[top_idx], probs[top_idx]

def run_gatekeeper(image_crop, target_name):
    """Kiểm tra vùng cắt: Là bệnh hay là nhiễu?"""
    # Nhãn đối ngẫu
    labels = [f"Pathological finding of {target_name}", "Normal tissue/Background noise"]
    
    image_tensor = preprocess(image_crop).unsqueeze(0).to(device)
    texts = tokenizer([f"this is {l}" for l in labels]).to(device)
    
    with torch.no_grad():
        image_features = model.encode_image(image_tensor)
        text_features = model.encode_text(texts)
        
        image_features /= image_features.norm(dim=-1, keepdim=True)
        text_features /= text_features.norm(dim=-1, keepdim=True)
        probs = (100.0 * image_features @ text_features.T).softmax(dim=-1)
        probs = probs.cpu().numpy()[0]
        
    # Index 0 là Pathological, Index 1 là Normal
    is_valid = probs[0] > 0.6  # Threshold
    return is_valid, probs[0]

In [7]:
import os
from PIL import Image
import matplotlib.pyplot as plt

# 1. Tạo thư mục images trên server nếu chưa có
os.makedirs("images", exist_ok=True)
image_path = os.path.join("images", "eval1.jpg")

# 2. Kiểm tra xem ảnh đã có chưa, nếu chưa thì cho upload
if not os.path.exists(image_path):
    print(f"⚠️ Không tìm thấy file tại '{image_path}'")
    print("👉 Đang mở widget upload... Vui lòng chọn file 'eval1.jpg' từ máy tính.")
    
    try:
        from google.colab import files
        uploaded = files.upload()
        
        # Di chuyển file vừa up vào folder images
        import shutil
        for filename in uploaded.keys():
            shutil.move(filename, os.path.join("images", filename))
            print(f"✅ Đã upload thành công: {filename}")
    except ImportError:
        print("❌ Bạn không chạy trên Colab. Hãy dùng Cách 1 (Kéo thả file vào VS Code).")

# 3. Load và hiển thị ảnh (Sau khi đã có file)
if os.path.exists(image_path):
    print(f"Loading image from: {image_path}")
    img = Image.open(image_path).convert("RGB")
    
    plt.imshow(img)
    plt.title("Input Image")
    plt.axis('off')
    plt.show()
else:
    print("❌ Vẫn chưa có file ảnh. Vui lòng kiểm tra lại.")

⚠️ Không tìm thấy file tại 'images/eval1.jpg'
👉 Đang mở widget upload... Vui lòng chọn file 'eval1.jpg' từ máy tính.


KeyboardInterrupt: 

In [5]:
# Chạy phân loại
detected_modality, conf = run_triage(img)

print(f"=== STEP 1: TRIAGE RESULT ===")
print(f"Detected: {detected_modality}")
print(f"Confidence: {conf:.2%}")

if conf > 0.8:
    print("-> High confidence. Injecting context into LLaVA...")
else:
    print("-> Low confidence. Fallback to general mode.")

NameError: name 'img' is not defined

In [ ]:
# Tự động tạo Box giả lập dựa trên kích thước ảnh thật
w, h = img.size

# Box 1: Giả lập vùng trung tâm (thường là tim/phổi/bệnh lý)
# Lấy vùng trung tâm ảnh (30% đến 70% chiều rộng/cao)
center_box = [int(w*0.3), int(h*0.3), int(w*0.7), int(h*0.7)]

# Box 2: Giả lập vùng góc trên bên trái (thường là nền đen/chữ/nhiễu)
corner_box = [0, 0, int(w*0.2), int(h*0.2)]

fake_boxes = [center_box, corner_box]

fig, ax = plt.subplots(1, figsize=(10, 10))
ax.imshow(img)

print("=== STEP 3: GATEKEEPER VERIFICATION ===")

# Giả định: Chúng ta đang tìm kiếm bất thường trong phổi
target_finding = "Lung Opacity" # Hoặc "Tumor", tùy ảnh

for i, box in enumerate(fake_boxes):
    # 1. Cắt ảnh (Crop)
    img_crop = img.crop((box[0], box[1], box[2], box[3]))
    
    # 2. Chạy Gatekeeper thật
    is_valid, score = run_gatekeeper(img_crop, target_finding)
    
    # 3. Vẽ hình minh họa
    color = 'lime' if is_valid else 'red'
    status = "KEEP (Pass)" if is_valid else "DROP (Fail)"
    
    # Vẽ khung
    rect = patches.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1], 
                             linewidth=3, edgecolor=color, facecolor='none')
    ax.add_patch(rect)
    
    # Ghi chú thích
    label_text = f"Box {i+1}: {status}\nScore: {score:.4f}"
    ax.text(box[0], box[1]-10, label_text, color=color, fontsize=12, weight='bold', 
            bbox=dict(facecolor='black', alpha=0.5))
    
    print(f"Checking Box {i+1} for '{target_finding}': Score {score:.4f} -> {status}")

plt.title(f"Gatekeeper Verification Result (Target: {target_finding})")
plt.axis('off')
plt.show()